## tl;dr

- 卒業ランク補正実装後の強いプレイでは、S率はハード2.6%、ノーマル44.7%、イージー54.4%。
- 奇数月の強イベントを毎回発生させる方式はノーマルS率79%まで上がったため不採用。従来の強イベント確率を保ち、残りを数値効果なしの日常イベントにした。
- 卒業生イベントは全難易度で平均2.00回発生し、3年次・4年次に各1回という仕様を満たした。

## Context & Methods

ショート版に、選択した卒業生5名からランダムに現れる先輩イベントと卒業ランク補正を追加した後の難易度を検証する。初期卒業生3名はBランクで、指導成功率+5%・成功時やる気+4。対象は実装済みゲームロジックで、直前の最良方針と今回のスモークテストで浮上した方針を含む4育成方針 × 4先輩選択方針から選定した。

### Key Assumptions

- 選定は各候補100シード、最終評価は未使用の1,000シード。
- 「最適」は全行動列の数学的最適解ではなく、定義した16候補内で平均ポイント最大の強いヒューリスティック。
- 強イベントは従来どおりキャライベント12.5%、ハプニング5%。残りの奇数月は数値効果なしの日常イベント。

## Data

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

data_path = Path.cwd() / 'docs/analysis/2026-07-23-alumni-balance.json'
report = json.loads(data_path.read_text(encoding='utf-8'))
report['definition']

In [ ]:
summary_rows = []
for cohort in report['cohorts']:
    summary_rows.append({
        '難易度': cohort['difficulty'],
        '採用方針': cohort['candidate']['alumniChoice'],
        'S率': cohort['rankRates']['S'],
        'S+A率': cohort['rankRates']['S'] + cohort['rankRates']['A'],
        '平均ポイント': cohort['meanPoints'],
        '平均能力': cohort['meanAbility'],
        '卒業生イベント回数': cohort['meanAlumniEvents'],
    })
summary = pd.DataFrame(summary_rows)
summary.style.format({'S率': '{:.1%}', 'S+A率': '{:.1%}', '平均ポイント': '{:.1f}', '平均能力': '{:.1f}', '卒業生イベント回数': '{:.2f}'})

## Results

In [ ]:
rank_order = ['S', 'A', 'B', 'C', 'D', 'E', '退学']
rank_rows = []
for cohort in report['cohorts']:
    row = {'難易度': cohort['difficulty']}
    row.update({rank: cohort['rankRates'][rank] for rank in rank_order})
    rank_rows.append(row)
rank_df = pd.DataFrame(rank_rows).set_index('難易度')[rank_order]
ax = rank_df.plot(kind='bar', stacked=True, figsize=(9, 4.8), width=0.72,
                  color=['#f5b642', '#ffcf66', '#73c6b6', '#5dade2', '#8e9aaf', '#c8c8c8', '#e76f51'])
ax.set_title('ショート版・難易度別の卒業ランク比率（各1,000シード）')
ax.set_xlabel('')
ax.set_ylabel('比率')
ax.set_ylim(0, 1)
ax.legend(title='ランク', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
checks = pd.DataFrame([
    {
        '難易度': cohort['difficulty'],
        '卒業率': cohort['graduationRate'],
        '卒業生イベント回数': cohort['meanAlumniEvents'],
        'AJDC総合優勝率': cohort['ajdcOverallWinRate'],
        '世界大会出場率': cohort['worldsQualifiedRate'],
        '世界大会優勝率': cohort['worldsWinRate'],
    }
    for cohort in report['cohorts']
])
checks.style.format({column: '{:.1%}' for column in ['卒業率', 'AJDC総合優勝率', '世界大会出場率', '世界大会優勝率']}
                    | {'卒業生イベント回数': '{:.2f}'})

## Takeaways

- ハードはS 2.6%で、成功すれば大きい挑戦枠のまま維持された。
- ノーマルはS 44.7%、S+A 68.1%。Bランク補正を加えても難易度差は維持された。
- イージーはS 54.4%、S+A 78.0%で、先に置いた暫定目標（S 45〜55%、S+A 70〜80%）に収まった。
- 先輩の選択は難易度で最適が変わる。ハード・ノーマルは得意技伝授、イージーは練習方法が選ばれ、3択が完全な一択にはなっていない。
- 次の検証は、自動育成AIも得意技対象へ練習を寄せる方針を追加し、ハイトス・FTSの実戦価値を再評価すること。